# Trading strategy and option pricing

This notebook compares a 20/50-day moving-average strategy with buy-and-hold on adjusted RELIANCE.NS closing prices. It includes a next-close execution delay, trading costs, risk metrics, and a separate Black-Scholes option-pricing example. Run it from the repository root.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

from trading_analysis import backtest_moving_average, performance_metrics, black_scholes, implied_volatility


In [ ]:
ticker = "RELIANCE.NS"
prices = yf.download(ticker, start="2023-01-01", auto_adjust=True, progress=False)
if prices.empty:
    raise ValueError(f"No prices returned for {ticker}")
close = prices["Close"]
if isinstance(close, pd.DataFrame):  # yfinance may use MultiIndex columns for one ticker
    close = close.iloc[:, 0]
close = close.rename("Close")
close.tail()


## Cost-aware moving-average backtest

Signals formed at the close fill at the following close. Returns begin after the fill, so the backtest cannot trade at a price it has already used to form the signal. A 5 bp cost is charged on each entry and exit; a long signal invests 50% of capital.


In [ ]:
results = backtest_moving_average(close, fast=20, slow=50, cost_bps=5, allocation=0.5)
summary = pd.Series(performance_metrics(results), name="Value").to_frame()
summary


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
results[["close", "fast_ma", "slow_ma"]].plot(ax=axes[0], title=f"{ticker}: price and moving averages")
results[["strategy_equity", "benchmark_equity"]].plot(ax=axes[1], title="Growth of 1 unit: strategy net of costs vs. buy-and-hold")
axes[1].set_ylabel("Portfolio value")
fig.tight_layout()
plt.show()


The summary is an in-sample historical description. Window selection should be evaluated on a separate out-of-sample period or with walk-forward testing before drawing conclusions about predictive value.


## Black-Scholes-Merton example

This is an illustrative European call with hypothetical inputs, independent of the Reliance price series. Rates and volatility are annual decimals; theta is per year, while vega and rho are per 1.0 input change.


In [ ]:
example = black_scholes(spot=100, strike=100, years=1, rate=0.05, volatility=0.20, option_type="call")
pd.Series(vars(example), name="Value")


In [ ]:
implied_volatility(
    market_price=example.price, spot=100, strike=100, years=1,
    rate=0.05, option_type="call"
)
